# Week 10

In [11]:
%pip install scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [13]:
# load preprocessed data
train_df = pd.read_csv("csv_files/preprocessed_train_video_games.csv")
test_df = pd.read_csv("csv_files/preprocessed_test_video_games.csv")
# load metadata
metadata_df = pd.read_parquet("meta_video_games.parquet")

In [14]:
# only use num item of test rows for the metadata
metadata_df = metadata_df[metadata_df['item_id'].isin(test_df['item_id'].unique())]
len(metadata_df)
print(metadata_df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 927 entries, 864 to 136488
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   item_id         927 non-null    object 
 1   title           927 non-null    object 
 2   description     927 non-null    object 
 3   features        927 non-null    object 
 4   categories      927 non-null    object 
 5   details         927 non-null    object 
 6   rating_number   927 non-null    int64  
 7   average_rating  927 non-null    float64
dtypes: float64(1), int64(1), object(6)
memory usage: 65.2+ KB
None


In [15]:
# show me one entire row of the metadata
metadata_df.iloc[0]


item_id                                                  B00000JRSB
title                               Final Fantasy VII - PlayStation
description       [Amazon.com, Long recognized as role-playing g...
features          [1 Player, RPG, 3 Disc Set, Excellent graphics...
categories        [Video Games, Legacy Systems, PlayStation Syst...
details           {'': None, 'AC Adapter Current': None, 'Access...
rating_number                                                  1631
average_rating                                                  4.7
Name: 864, dtype: object

Description, features, categories and average rating is what will I use. Because I feel these have the most valuable information for comparision between items.

In [16]:
# text preprocessing
from sklearn.feature_extraction.text import TfidfVectorizer
# turn this into a function for any column of text data
def preprocess_text_column(df, column_name):
    # convert the column to string type (in case there are any missing values)
    df[column_name] = df[column_name].fillna('').astype(str)
    # lower case
    df[column_name] = df[column_name].str.lower()
    # remove punctuation
    df[column_name] = df[column_name].str.replace('[^\\w\\s]', '', regex=True)
    return df

def tfidf_vectorizing(df, column_name):
    # create a tf-idf vectorizer
    tfidf_vectorizer = TfidfVectorizer(stop_words='english')
    # fit and transform the metadata text data into tf-idf vectors
    tfidf_matrix = tfidf_vectorizer.fit_transform(df[column_name])
    # did we tokenize?
    return tfidf_vectorizer, tfidf_matrix

# preprocess the description column
metadata_df = preprocess_text_column(metadata_df, 'description')
# vectorize the description column (do not overwrite metadata_df)
tfidf_vectorizer, tfidf_matrix = tfidf_vectorizing(metadata_df, 'description')
# check vocabulary size
print(f"Vocabulary size: {len(tfidf_vectorizer.get_feature_names_out())}")

Vocabulary size: 18440


In [17]:
# call the same fucntions on the features column
metadata_df = preprocess_text_column(metadata_df, 'features')
tfidf_vectorizer_features, tfidf_matrix_features = tfidf_vectorizing(metadata_df, 'features')
print(f"Vocabulary size for features: {len(tfidf_vectorizer_features.get_feature_names_out())}")

Vocabulary size for features: 9127


In [18]:
# call the functions for category column
metadata_df = preprocess_text_column(metadata_df, 'categories')
tfidf_vectorizer_category, tfidf_matrix_category = tfidf_vectorizing(metadata_df, 'categories')
print(f"Vocabulary size for category: {len(tfidf_vectorizer_category.get_feature_names_out  ())}")

Vocabulary size for category: 80


same preprocessing used on the categories and features columns

Use Normalized Weighted Mean:
Normalized Weighted Mean: To reduce bias, the mean feature value across all items is subtracted from the weighted average, highlighting what makes a user's tastes unique compared to the average

In [21]:
from scipy.sparse import hstack, csr_matrix, diags
import numpy as np

# incorporate average item rating as an additional feature
avg_ratings = (
    train_df.groupby("item_id", as_index=False)["rating"]
    .mean()
    .rename(columns={"rating": "avg_item_rating"})
)

# make this cell re-runnable without creating duplicate/suffixed columns
metadata_df = metadata_df.drop(columns=["avg_item_rating"], errors="ignore")
metadata_df = metadata_df.merge(avg_ratings, on="item_id", how="left")
metadata_df["avg_item_rating"] = metadata_df["avg_item_rating"].fillna(0.0)

# sparse numeric feature for item average rating
tfidf_matrix_rating = csr_matrix(metadata_df["avg_item_rating"].to_numpy().reshape(-1, 1))

# 1) Build a single item-vector space (description + features + categories + avg_item_rating)
item_matrix = hstack([tfidf_matrix, tfidf_matrix_features, tfidf_matrix_category, tfidf_matrix_rating]).tocsr()

# Map item_id -> row index in item_matrix
item_id_to_row = pd.Series(np.arange(len(metadata_df)), index=metadata_df["item_id"]).to_dict()

# 2) Keep only interactions for items that exist in metadata_df/item_matrix
interactions = train_df[train_df["item_id"].isin(item_id_to_row)].copy()
interactions["item_row"] = interactions["item_id"].map(item_id_to_row).astype(np.int32)

# 3) Create user-item matrix weighted by rating
user_ids = interactions["user_id"].unique()
user_id_to_row = pd.Series(np.arange(len(user_ids)), index=user_ids).to_dict()
interactions["user_row"] = interactions["user_id"].map(user_id_to_row).astype(np.int32)

R = csr_matrix(
    (
        interactions["rating"].values,
        (interactions["user_row"].values, interactions["item_row"].values),
    ),
    shape=(len(user_ids), len(metadata_df)),
)

# normalized weighted mean - row-normalize by the sum of ratings for each user
row_sums = np.array(R.sum(axis=1)).flatten()
row_sums[row_sums == 0] = 1.0
D_inv = diags(1.0 / row_sums)

# 4) User vectors in the same space as item vectors
user_matrix = (D_inv @ R @ item_matrix).tocsr()

print("Item matrix shape:", item_matrix.shape)
print("User matrix shape:", user_matrix.shape)
print("Users represented:", len(user_ids))

Item matrix shape: (927, 27648)
User matrix shape: (1389, 27648)
Users represented: 1389


I am a bit confused we have these user and item matrices, but how do we use them to predict ratings?  
the cosine similarity between the user vector and item vector gives us a measure of how well the item matches the   
user's preferences based on their past interactions. But how was the user vector constructed?   
It is a weighted average of the item vectors for the items that the user has interacted with,   
where the weights are based on the user's interactions (in this case, we used equal weights).  
So if a user has interacted with items that are similar to a new item, 
their user vector will be similar to that item's vector,  
resulting in a higher cosine similarity and thus a higher predicted rating.
We can then map this similarity score to a rating scale (e.g., 1 to 5) to get a predicted rating for that user-item pair.

In [22]:
# Cosine-similarity-based user-item rating prediction

# 1) Helper: predict one (user_id, item_id) pair
def predict_rating_cosine(user_id, item_id, min_rating=1.0, max_rating=5.0):
    u = user_id_to_row.get(user_id)
    i = item_id_to_row.get(item_id)
    if u is None or i is None:
        return np.nan  # unseen user or item

    u_vec = user_matrix[u]
    i_vec = item_matrix[i]

    dot_ui = u_vec.multiply(i_vec).sum()
    u_norm = np.sqrt(u_vec.multiply(u_vec).sum())
    i_norm = np.sqrt(i_vec.multiply(i_vec).sum())

    if u_norm == 0 or i_norm == 0:
        return np.nan

    cos_sim = float(dot_ui / (u_norm * i_norm))
    cos_sim = np.clip(cos_sim, 0.0, 1.0)  # keep in [0,1]

    # map cosine similarity to rating scale [1,5]
    pred_rating = min_rating + (max_rating - min_rating) * cos_sim
    return pred_rating

# use the cosine similarity prediction function to predict ratings for the train set
interactions["pred_rating_cosine"] = interactions.apply(
    lambda row: predict_rating_cosine(row["user_id"], row["item_id"]), axis=1
)
# having a hard time keeping track of the datastrucutres and types used
# but we can check the predictions for a few user-item pairs to see if they look reasonable
print(interactions[["user_id", "item_id", "rating", "pred_rating_cosine"]].head(10))
# rating RMSE and MAE for the predicitons
from sklearn.metrics import mean_squared_error, mean_absolute_error
valid_preds = interactions.dropna(subset=["pred_rating_cosine"])
rmse = np.sqrt(mean_squared_error(valid_preds["rating"], valid_preds["pred_rating_cosine"]))
mae = mean_absolute_error(valid_preds["rating"], valid_preds["pred_rating_cosine"])
print(f"Cosine Similarity Prediction - RMSE: {rmse:.4f}, MAE: {mae:.4f}")


                        user_id     item_id  rating  pred_rating_cosine
0  AHOGCWGRSFQ6YZH6QLYUMNQ4N3KA  B00ZM5OXD8     5.0            4.792916
1  AEKOQRDUY64SGH4PONBFTSIM2I2Q  B00O3JSRHW     4.0            4.729520
2  AFA43JCV3C72LHM5BIVPV7UEJ2CA  B0056WJA30     5.0            4.813053
3  AFD3QCEDODUYBYMQGA6NWILJW7KA  B002I0J4NE     5.0            4.827546
4  AGYF4ZSMTSZHCE3OH6CO5SJK5Y3A  B0031SWWPO     5.0            4.709997
5  AEQLEOC6QXNCA6PEA6JA2FNSFLDQ  B09V5R5LSZ     5.0            4.763791
6  AHJFYPQ3PUPX5FAK2VGWORJF3UAQ  B0013RATNM     5.0            4.787075
7  AEDZDDJ72FW2KTJ6PRY3HXOZQPSQ  B000ZK7ZOE     4.0            4.826369
8  AGUYMMFXOZMSJJUET7RN2A6O5AJQ  B004HD55VK     4.0            4.820082
9  AHR5PQR4OH6P7RKLJRO7D3RUJFUQ  B002I0K6DG     5.0            4.811726
Cosine Similarity Prediction - RMSE: 1.2559, MAE: 0.7966


In [24]:
# evaluate predictions to test set ratings
# Report Precision@10, MAP@10, MRR@10, Hit Rate and Coverage using ratings >= 3 as relevant 
# hit rate also uses relevancy

k = 10
relevance_threshold = 3.0

# Keep only test pairs we can score in current matrices
eval_test = test_df[
    test_df["user_id"].isin(user_id_to_row) & test_df["item_id"].isin(item_id_to_row)
].copy()
eval_test["item_row"] = eval_test["item_id"].map(item_id_to_row)

# Users to evaluate (known users in test)
eval_users = np.array(eval_test["user_id"].unique())
eval_user_rows = np.array([user_id_to_row[u] for u in eval_users], dtype=np.int32)

# Relevant items per user from test set (rating >= 3)
relevant_df = eval_test[eval_test["rating"] >= relevance_threshold]
relevant_by_user = relevant_df.groupby("user_id")["item_row"].apply(set).to_dict()

# Items already seen in train (exclude from recommendation)
seen_by_user = interactions.groupby("user_id")["item_row"].apply(set).to_dict()

# Cosine scores for all eval users vs all items
U = user_matrix[eval_user_rows]
dot_scores = (U @ item_matrix.T).toarray()

u_norms = np.sqrt(U.multiply(U).sum(axis=1)).A1
i_norms = np.sqrt(item_matrix.multiply(item_matrix).sum(axis=1)).A1
den = np.outer(np.where(u_norms == 0, 1.0, u_norms), np.where(i_norms == 0, 1.0, i_norms))
scores = dot_scores / den

# Build top-k recommendations per user
topk_by_user = {}
all_recommended_items = set()
n_items = item_matrix.shape[0]

for idx, u in enumerate(eval_users):
    s = scores[idx].copy()

    # exclude training-history items
    seen_items = seen_by_user.get(u, set())
    if seen_items:
        s[list(seen_items)] = -np.inf

    finite_idx = np.where(np.isfinite(s))[0]
    if finite_idx.size == 0:
        recs = np.array([], dtype=np.int32)
    else:
        kk = min(k, finite_idx.size)
        cand = np.argpartition(-s[finite_idx], kk - 1)[:kk]
        recs = finite_idx[cand]
        recs = recs[np.argsort(-s[recs])]

    topk_by_user[u] = recs
    all_recommended_items.update(recs.tolist())

# Ranking metrics
precisions, ap_scores, rr_scores, hit_rates = [], [], [], []

for u in eval_users:
    recs = topk_by_user[u]
    rel = relevant_by_user.get(u, set())

    hits = np.array([1 if r in rel else 0 for r in recs], dtype=np.int32)

    # Precision@k
    precisions.append(hits.sum() / k)

    # Hit Rate@k
    hit_rates.append(1.0 if hits.sum() > 0 else 0.0)

    # AP@k and MRR@k (only defined for users with at least 1 relevant item in test)
    if len(rel) > 0:
        if hits.sum() > 0:
            cum_hits = np.cumsum(hits)
            hit_pos = np.where(hits == 1)[0]
            ap = (cum_hits[hit_pos] / (hit_pos + 1)).sum() / min(len(rel), k)
            rr = 1.0 / (hit_pos[0] + 1)
        else:
            ap, rr = 0.0, 0.0
        ap_scores.append(ap)
        rr_scores.append(rr)

precision_at_10 = float(np.mean(precisions)) if precisions else np.nan
map_at_10 = float(np.mean(ap_scores)) if ap_scores else np.nan
mrr_at_10 = float(np.mean(rr_scores)) if rr_scores else np.nan
hit_rate_at_10 = float(np.mean(hit_rates)) if hit_rates else np.nan
coverage = float(len(all_recommended_items) / n_items) if n_items > 0 else np.nan

print("Content based recommendation system using description, feature, categories,and ratings with normalized weight mean for user vectors")
print(f"Users evaluated: {len(eval_users)}")
print(f"Precision@10: {precision_at_10:.4f}")
print(f"MAP@10:       {map_at_10:.4f}")
print(f"MRR@10:       {mrr_at_10:.4f}")
print(f"Hit Rate@10:  {hit_rate_at_10:.4f}")
print(f"Coverage:     {coverage:.4f}")


Content based recommendation system using description, feature, categories,and ratings with normalized weight mean for user vectors
Users evaluated: 1389
Precision@10: 0.0084
MAP@10:       0.0045
MRR@10:       0.0197
Hit Rate@10:  0.0828
Coverage:     0.0831
